In [16]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [17]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [18]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [19]:
# shuffle up the words
import random
random.seed(42)
random.shuffle(words)

In [20]:
# build the dataset
block_size = 16 # context length: how many characters do we take to predict the next one?

def build_dataset(words):
  X, Y = [], []

  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

n1 = int(0.8*len(words))
n2 = int(0.9*len(words))
Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

torch.Size([182625, 16]) torch.Size([182625])
torch.Size([22655, 16]) torch.Size([22655])
torch.Size([22866, 16]) torch.Size([22866])


In [21]:
for x,y in zip(Xtr[:20], Ytr[:20]):
  print(''.join(itos[ix.item()] for ix in x), '-->', itos[y.item()])

................ --> y
...............y --> u
..............yu --> h
.............yuh --> e
............yuhe --> n
...........yuhen --> g
..........yuheng --> .
................ --> d
...............d --> i
..............di --> o
.............dio --> n
............dion --> d
...........diond --> r
..........diondr --> e
.........diondre --> .
................ --> x
...............x --> a
..............xa --> v
.............xav --> i
............xavi --> e


In [22]:
torch.manual_seed(42); # seed rng for reproducibility

In [23]:
from nn import Sequential, Embedding, FlattenConsecutive, Linear, BatchNorm1d, Tanh

In [24]:
# original network
# n_embd = 10 # the dimensionality of the character embedding vectors
# n_hidden = 200 # the number of neurons in the hidden layer of the MLP
# model = Sequential([
#   Embedding(vocab_size, n_embd),# 27 x 10
#   Flatten(),
#   Linear(n_embd * block_size, n_hidden, bias=False),
#   BatchNorm1d(n_hidden),
#   Tanh(),
#   Linear(n_hidden, vocab_size),
# ])
#
# # parameter init
# with torch.no_grad():
#   model.layers[-1].weight *= 0.1 # last layer make less confident
#
# parameters = model.parameters()
# print(sum(p.nelement() for p in parameters)) # total number of parameters
# for p in parameters:
#   p.requires_grad = True


In [25]:
# hierarchical network
n_embd = 24 # the dimensionality of the character embedding vectors
n_hidden = 128 # the number of neurons in the hidden layer of the MLP
model = Sequential([
  Embedding(vocab_size, n_embd),
  FlattenConsecutive(2), Linear(n_embd * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  FlattenConsecutive(2), Linear(n_hidden*2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  FlattenConsecutive(2), Linear(n_hidden*2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  FlattenConsecutive(2), Linear(n_hidden*2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
  Linear(n_hidden, vocab_size),
])

# parameter init
with torch.no_grad():
  model.layers[-1].weight *= 0.1 # last layer make less confident

parameters = model.parameters()
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

109603


In [26]:
# ix = torch.randint(0, Xtr.shape[0], (4,))
# Xb, Yb = Xtr[ix], Ytr[ix]
# logits = model(Xb)
# ix, Xb.shape, Xb


In [27]:
# for layer in model.layers:
#   print(layer.__class__.__name__, ':', tuple(layer.out.shape))

In [28]:
# e = torch.randn(4, 8, 10)
# exp_e = torch.cat([e[:, ::2, :], e[:, 1::2, :]], dim=2)

In [ ]:
# same optimization as last time
max_steps = 200000
batch_size = 32
lossi = []
print([p.requires_grad for p in parameters])
if torch.is_grad_enabled() == False:
  print("Gradient calculation is disabled")
  torch.set_grad_enabled(True)

for i in range(max_steps):

  # minibatch construct
  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  Xb, Yb = Xtr[ix], Ytr[ix] # batch X,Y

  # forward pass
  logits = model(Xb)
  loss = F.cross_entropy(logits, Yb) # loss function

  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()

  # update: simple SGD
  lr = 0.1 if i < 150000 else 0.01 # step learning rate decay
  for p in parameters:
    p.data += -lr * p.grad

  # track stats
  if i % 10000 == 0: # print every once in a while
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')

  lossi.append(loss.log10().item())

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
Gradient calculation is disabled
      0/ 200000: 3.2724
  10000/ 200000: 2.2082
  20000/ 200000: 2.3744
  30000/ 200000: 1.8629
  40000/ 200000: 1.8262
  50000/ 200000: 2.0641
  60000/ 200000: 1.6649
  70000/ 200000: 1.8008
  80000/ 200000: 1.8449
  90000/ 200000: 2.0763
 100000/ 200000: 1.9319
 110000/ 200000: 1.9211
 120000/ 200000: 1.7119
 130000/ 200000: 1.6609
 140000/ 200000: 1.8649


In [ ]:
for layer in model.layers:
  layer.training = False

In [ ]:
@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  logits = model(x)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')
# split_loss('test')

In [ ]:
# sample from the model
for _ in range(20):

    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      # forward pass the neural net
      ctx = torch.tensor([context]) # (1,block_size,n_embd)
      logits = model(ctx) # (1,block_size,vocab_size)
      probs = F.softmax(logits, dim=1)
      # sample from the distribution
      ix = torch.multinomial(probs, num_samples=1).item()
      # shift the context window and track the samples
      context = context[1:] + [ix]
      out.append(ix)
      # if we sample the special '.' token, break
      if ix == 0:
        break

    print(''.join(itos[i] for i in out)) # decode and print the generated word

In [ ]:
plt.plot(torch.tensor(lossi).view(-1, 1000).mean(1))